# Do the saved models follow their Euclidean choice rule?

This notebook reloads the fitted models and evaluates every held-out comparison. It checks whether swapping alternatives complements the probability, whether the nearer proposal is favored, and whether the saved coordinates agree with the weights. It does not train a model.

**Inputs:** the filtered pairs and Chilean split in `data/processed/`, plus the weights, specifications, coordinates and recorded accuracy in `outputs/models/`. These files are included. Notebooks 01–04 show how to regenerate them.

**Outputs:** sample sizes and prediction diagnostics in `outputs/diagnostics/choice_geometry/`, shown below and exported as CSV and LaTeX tables.

**Explore:** run the notebook from top to bottom. To evaluate fewer Chilean initializations, edit `CHILE_SEEDS` below. To evaluate another fitted model, change the model directory and fit list. A model specification is metadata saved with its weights, so it is read rather than typed again.

A model favoring the farther alternative is different from a participant choosing the farther alternative. These checks concern the model's probabilities. They do not establish geometric stability or put independently fitted countries on a common scale.


For another supplied Chilean dimension, change `CHILE_DIMENSION` and select seeds that have saved fits in that dimension. The diagnostics read all coordinate columns from each model specification. France and Brazil use their supplied two-dimensional fits.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/raw").is_dir(), "Open from the project root or notebooks directory."
sys.path.insert(0, str(ROOT / "src"))
from choice_models import build_choice_model

PROCESSED = ROOT / "data/processed"
MODELS = ROOT / "outputs/models"
DIAGNOSTICS = ROOT / "outputs/diagnostics/choice_geometry"
DIAGNOSTICS.mkdir(parents=True, exist_ok=True)

CHILE_DIMENSION = 2
CHILE_SEEDS = [101, 201, 301, 401, 501]
BATCH_SIZE = 8192  # Inference batches only, independent of the training batch size.
LOGLOSS_CLIP = 1e-7
# Treat rounding-sized differences as unresolved ties, not geometric failures.
PROBABILITY_TOLERANCE = 2 * np.finfo(np.float32).eps
DISTANCE_ERROR_MULTIPLIER = 8 * np.finfo(np.float32).eps
COORDINATE_TOLERANCE = 2 * np.finfo(np.float32).eps


## Load the specifications of the fits being evaluated

Each specification records the numbers of embeddings, coordinate dimension, neural architecture and initialization seeds. Reconstructing that architecture before loading its weights prevents a saved fit from being silently interpreted as another model. The table below lists only the fits evaluated here.


In [2]:
fits = [
    {"country": "Chile", "seed": seed, "run_name": f"Chile_dim{CHILE_DIMENSION}_seed{seed}"}
    for seed in CHILE_SEEDS
]
fits += [
    {"country": "France", "seed": 301, "run_name": "france"},
    {"country": "Brazil", "seed": 101, "run_name": "brazil"},
]
saved_specifications = {}
for fit in fits:
    path = MODELS / fit["run_name"] / "model_specification.json"
    saved_specifications[fit["run_name"]] = json.loads(path.read_text())

model_inventory = pd.DataFrame(
    [
        {
            "fit": name,
            "variant": value["variant"],
            "dimensions": value["dimension"],
            "seed": value["seed"],
            "participants": value["n_users"],
            "proposals": value["n_options"],
        }
        for name, value in saved_specifications.items()
    ]
)
display(model_inventory)
model_inventory.to_csv(DIAGNOSTICS / "model_inventory.csv", index=False)
model_inventory.to_latex(DIAGNOSTICS / "model_inventory.tex", index=False)


,fit,variant,dimensions,seed,participants,proposals
0,Chile_dim2_seed101,intermediate,2,101,47574,90
1,Chile_dim2_seed201,intermediate,2,201,47574,90
2,Chile_dim2_seed301,intermediate,2,301,47574,90
3,Chile_dim2_seed401,intermediate,2,401,47574,90
4,Chile_dim2_seed501,intermediate,2,501,47574,90
5,france,intermediate,2,301,1443,120
6,brazil,intermediate,2,101,765,67


## Reconstruct the held-out inputs

Chile uses its saved split after the stable ordering used during training. France and Brazil use the first 90% of each person's records for training, following the original retained order. A person with one record contributes it to training. The remaining comparisons are evaluated here without shuffling or resampling.

The neural layer expects integer indices, not the source identifiers. Sorted participant and proposal IDs reproduce the mapping used during training. Each input row is `[proposal A, participant, proposal B]`. Its target is one when A was selected. The checks below ensure that every evaluated participant has training support and that age exclusions remain satisfied.


In [3]:
age = pd.read_csv(PROCESSED / "participant_age_decisions.csv", dtype={"uuid": str})
assert not age.duplicated(["country", "uuid"]).any()
include = (
    age["include"]
    .astype(str)
    .str.lower()
    .map({"true": True, "false": False, "1": True, "0": False})
)
assert include.notna().all()
age["include"] = include.astype(bool)
samples = {}
sample_rows = []

for country in ["Chile", "France", "Brazil"]:
    if country == "Chile":
        source = pd.read_parquet(PROCESSED / "Chile_pairs.parquet")
        assignments = pd.read_parquet(PROCESSED / "Chile_split_assignments.parquet")
        assert source["id"].is_unique and assignments["id"].is_unique
        assert len(source) == len(assignments)
        source = source.sort_values(["uuid", "id", "datetime"], kind="mergesort").reset_index(
            drop=True
        )
        source = source.merge(
            assignments, on=["id", "uuid"], how="left", validate="one_to_one", sort=False
        )
        assert source["split"].isin(["train", "test"]).all()
        user_key, record_key = "uuid", "id"
        train_indices = np.flatnonzero(source["split"].eq("train").to_numpy())
        test_indices = np.flatnonzero(source["split"].eq("test").to_numpy())
        target_dtype = np.int32
    else:
        source = pd.read_parquet(PROCESSED / f"{country.lower()}_pairwise.parquet")
        assert source.index.equals(pd.RangeIndex(len(source)))
        user_key, record_key = "user_id", "id_original"
        source[user_key] = source[user_key].astype(str)
        train_indices, test_indices = [], []
        for _, participant_rows in source.groupby(user_key, sort=True):
            indices = participant_rows.index.to_numpy()
            # floor(0.9*n) matches notebook 03, except that a singleton stays in training.
            cutoff = 1 if len(indices) == 1 else int(len(indices) * 0.9)
            train_indices.extend(indices[:cutoff])
            test_indices.extend(indices[cutoff:])
        train_indices = np.asarray(train_indices, dtype=np.int64)
        test_indices = np.asarray(test_indices, dtype=np.int64)
        target_dtype = np.float32
    source[user_key] = source[user_key].astype(str)
    assert source[record_key].is_unique
    assert (
        (source["selected"] == source["option_a"]) | (source["selected"] == source["option_b"])
    ).all()
    participant_ids = sorted(source[user_key].unique())
    proposal_ids = sorted(set(source["option_a"]) | set(source["option_b"]))
    # Keep the index order identical to the embedding tables saved in training.
    participant_index = {value: index for index, value in enumerate(participant_ids)}
    proposal_index = {value: index for index, value in enumerate(proposal_ids)}
    X = np.column_stack(
        [
            source["option_a"].map(proposal_index),
            source[user_key].map(participant_index),
            source["option_b"].map(proposal_index),
        ]
    ).astype(np.int32)
    y = source["selected"].eq(source["option_a"]).to_numpy(dtype=target_dtype)
    assert len(train_indices) + len(test_indices) == len(source)
    assert len(np.intersect1d(train_indices, test_indices)) == 0
    assert set(X[train_indices, 1]) == set(range(len(participant_ids)))
    assert set(X[test_indices, 1]).issubset(set(X[train_indices, 1]))
    # Check participant-level eligibility after reconstructing the exact evaluation sample.
    country_age = age.loc[age["country"].str.casefold().eq(country.casefold())].set_index(
        "uuid"
    )
    assert set(participant_ids).issubset(country_age.index)
    assert country_age.loc[participant_ids, "include"].all()
    test_X, test_y = X[test_indices], y[test_indices]
    samples[country] = dict(
        X_test=test_X,
        y_test=test_y,
        participant_ids=participant_ids,
        proposal_ids=proposal_ids,
        user_key=user_key,
        records=len(source),
        training_records=len(train_indices),
    )
    sample_rows.append(
        {
            "country": country,
            "participants": len(participant_ids),
            "proposals": len(proposal_ids),
            "records": len(source),
            "training_records": len(train_indices),
            "heldout_records": len(test_y),
        }
    )
    del source, X, y

sample_summary = pd.DataFrame(sample_rows)
display(sample_summary)
sample_summary.to_csv(DIAGNOSTICS / "heldout_samples.csv", index=False)
sample_summary.to_latex(DIAGNOSTICS / "heldout_samples.tex", index=False)


,country,participants,proposals,records,training_records,heldout_records
0,Chile,47574,90,2970347,2650103,320244
1,France,1443,120,1983640,1784636,199004
2,Brazil,765,67,163803,147106,16697


## Evaluate probabilities and compare them with distances

Accuracy uses `P(A) > 0.5`, matching the training metric. Log loss clips probabilities to avoid taking the logarithm of zero. Clipping affects that diagnostic only.

The embedding layer stores float32 coordinates. Distances are recomputed in float64, and the coordinate CSVs are checked against the actual weights before evaluation. For a comparison, `s_b - s_a > 0` means A is nearer. The intermediate model must then assign A at least half the probability. Swapping the A and B index columns should give the complementary probability.

A distance difference within `8 * float32_epsilon * max(1, s_a+s_b)` is treated as unresolved. A probability within two float32 epsilons of 0.5 is also unresolved. These rules prevent rounding-sized differences from appearing as violations. They do not remove any record from accuracy or log loss.


In [4]:
rows = []
for fit in fits:
    country = fit["country"]
    sample = samples[country]
    run = MODELS / fit["run_name"]
    model_config = saved_specifications[fit["run_name"]]
    assert model_config["seed"] == fit["seed"]
    assert model_config["n_users"] == len(sample["participant_ids"])
    assert model_config["n_options"] == len(sample["proposal_ids"])
    weight_path = run / "model.weights.h5"
    tf.keras.backend.clear_session()
    model = build_choice_model(**model_config)
    model.load_weights(weight_path)
    # Extract coordinates from the reloaded weights, then compare with their CSV exports.
    users = model.layers[0].returnUsersEmbedding().numpy().astype(np.float64)
    proposals = model.layers[0].returnOptionsEmbedding().numpy().astype(np.float64)
    columns = [f"z{i+1}" for i in range(model_config["dimension"])]
    user_csv = pd.read_csv(
        run / "coordsU.csv", dtype={sample["user_key"]: str}, float_precision="round_trip"
    )
    proposal_key = "id" if country == "Chile" else "option_id"
    proposal_csv = pd.read_csv(run / "coordsP.csv", float_precision="round_trip")
    assert user_csv[sample["user_key"]].tolist() == sample["participant_ids"]
    assert proposal_csv[proposal_key].tolist() == sample["proposal_ids"]
    user_error = float(np.max(np.abs(user_csv[columns].to_numpy() - users)))
    proposal_error = float(np.max(np.abs(proposal_csv[columns].to_numpy() - proposals)))
    assert max(user_error, proposal_error) <= COORDINATE_TOLERANCE
    X, y = sample["X_test"], sample["y_test"]
    predicted, swapped = [], []
    for start in range(0, len(X), BATCH_SIZE):
        batch = X[start : start + BATCH_SIZE]
        predicted.append(model(batch, training=False).numpy().ravel())
        # [2, 1, 0] exchanges alternatives while keeping the participant fixed.
        swapped.append(model(batch[:, [2, 1, 0]], training=False).numpy().ravel())
    probability = np.concatenate(predicted).astype(np.float64)
    reverse_probability = np.concatenate(swapped).astype(np.float64)
    assert np.isfinite(probability).all() and np.isfinite(reverse_probability).all()
    assert ((probability >= 0) & (probability <= 1)).all()
    accuracy = float(np.mean((probability > 0.5) == y))
    clipped = np.clip(probability, LOGLOSS_CLIP, 1 - LOGLOSS_CLIP)
    log_loss = float(-np.mean(y * np.log(clipped) + (1 - y) * np.log1p(-clipped)))
    swap_error = np.abs(probability + reverse_probability - 1)
    s_a = np.sum((proposals[X[:, 0]] - users[X[:, 1]]) ** 2, axis=1)
    s_b = np.sum((proposals[X[:, 2]] - users[X[:, 1]]) ** 2, axis=1)
    distance_delta = s_b - s_a  # Positive means A is closer.
    distance_tolerance = DISTANCE_ERROR_MULTIPLIER * np.maximum(1, s_a + s_b)
    distance_direction = np.where(
        distance_delta > distance_tolerance,
        1,
        np.where(distance_delta < -distance_tolerance, -1, 0),
    )
    probability_direction = np.where(
        probability > 0.5 + PROBABILITY_TOLERANCE,
        1,
        np.where(probability < 0.5 - PROBABILITY_TOLERANCE, -1, 0),
    )
    # Opposite nonzero signs mean the model favors the farther proposal.
    both_resolved = (distance_direction != 0) & (probability_direction != 0)
    farther = (distance_direction * probability_direction) < 0
    raw_farther = distance_delta * (probability - 0.5) < 0
    if model_config["variant"] == "intermediate":
        assert not farther.any(), f"Resolved Euclidean contradiction in {fit['run_name']}"
        assert float(swap_error.max()) <= 4 * np.finfo(np.float32).eps
    if country == "Chile":
        recorded = pd.read_csv(run / "metrics.csv").iloc[0]
        recorded_accuracy = float(recorded["test_accuracy_all_records"])
    else:
        properties = json.loads((run / "model_properties.json").read_text())
        recorded_accuracy = float(properties["test_accuracy"])
    assert abs(recorded_accuracy - accuracy) <= 1e-6
    row = {
        "country": country,
        "run_name": fit["run_name"],
        "variant": model_config["variant"],
        "dimension": model_config["dimension"],
        "seed": fit["seed"],
        "heldout_records": len(y),
        "accuracy": accuracy,
        "clipped_log_loss": log_loss,
        "log_loss_clip": LOGLOSS_CLIP,
        "closer_rule_accuracy": float(np.mean((distance_delta > 0) == y)),
        "raw_farther_count": int(raw_farther.sum()),
        "resolved_farther_count": int(farther.sum()),
        "resolved_comparisons": int(both_resolved.sum()),
        "farther_fraction_all_records": float(farther.mean()),
        "farther_fraction_resolved": (
            float(farther.sum() / both_resolved.sum()) if both_resolved.any() else None
        ),
        "distance_tie_count": int((distance_direction == 0).sum()),
        "probability_tie_count": int((probability_direction == 0).sum()),
        "saturated_probability_count": int(((probability == 0) | (probability == 1)).sum()),
        "mean_swap_error": float(swap_error.mean()),
        "maximum_swap_error": float(swap_error.max()),
        "maximum_coordinate_csv_error": max(user_error, proposal_error),
    }
    rows.append(row)
    print(f"{fit['run_name']}: {len(y):,} held-out records evaluated.")


Chile_dim2_seed101: 320,244 held-out records evaluated.


Chile_dim2_seed201: 320,244 held-out records evaluated.


Chile_dim2_seed301: 320,244 held-out records evaluated.


Chile_dim2_seed401: 320,244 held-out records evaluated.


Chile_dim2_seed501: 320,244 held-out records evaluated.


france: 199,004 held-out records evaluated.
brazil: 16,697 held-out records evaluated.


## Read the results

`accuracy` measures agreement with actual choices. `resolved_farther_count` instead counts model predictions that contradict the distance ordering beyond numerical tolerances. It should be zero for the intermediate architecture. `maximum_swap_error` measures the largest departure from complementary probabilities after exchanging the proposals.

The full table retains ties, saturation and coordinate-export errors. The shorter comparison shows the main checks across fits. Stability of the locations themselves is examined separately in notebook 14.


In [5]:
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(DIAGNOSTICS / "choice_geometry_summary.csv", index=False)
summary.to_latex(DIAGNOSTICS / "choice_geometry_summary.tex", index=False)
comparison = summary[
    [
        "country",
        "seed",
        "accuracy",
        "clipped_log_loss",
        "resolved_farther_count",
        "distance_tie_count",
        "probability_tie_count",
        "maximum_swap_error",
    ]
].copy()
display(comparison)
comparison.to_csv(DIAGNOSTICS / "choice_geometry_comparison.csv", index=False)
comparison.to_latex(
    DIAGNOSTICS / "choice_geometry_comparison.tex", index=False, float_format="%.6f"
)


,country,run_name,variant,dimension,seed,heldout_records,accuracy,clipped_log_loss,log_loss_clip,closer_rule_accuracy,...,resolved_farther_count,resolved_comparisons,farther_fraction_all_records,farther_fraction_resolved,distance_tie_count,probability_tie_count,saturated_probability_count,mean_swap_error,maximum_swap_error,maximum_coordinate_csv_error
0,Chile,Chile_dim2_seed101,intermediate,2,101,320244,0.680987,0.594979,1.000000e-07,0.680987,...,0,320241,0.0,0.0,3,0,0,2.142049e-08,8.940697e-08,2.980194e-08
1,Chile,Chile_dim2_seed201,intermediate,2,201,320244,0.684375,0.593649,1.000000e-07,0.684375,...,0,320240,0.0,0.0,4,1,1,2.113969e-08,1.043081e-07,2.980194e-08
2,Chile,Chile_dim2_seed301,intermediate,2,301,320244,0.686483,0.589048,1.000000e-07,0.686483,...,0,320244,0.0,0.0,0,0,0,2.114503e-08,1.192093e-07,2.980042e-08
3,Chile,Chile_dim2_seed401,intermediate,2,401,320244,0.682242,0.597383,1.000000e-07,0.682267,...,0,320183,0.0,0.0,61,59,0,2.122617e-08,1.192093e-07,2.980194e-08
4,Chile,Chile_dim2_seed501,intermediate,2,501,320244,0.685374,0.592766,1.000000e-07,0.685356,...,0,320178,0.0,0.0,66,64,0,2.106406e-08,8.940697e-08,2.979889e-08
5,France,france,intermediate,2,301,199004,0.627043,0.761073,1.000000e-07,0.627043,...,0,199002,0.0,0.0,2,0,0,1.968829e-08,1.192093e-07,2.977524e-08
6,Brazil,brazil,intermediate,2,101,16697,0.629215,0.743122,1.000000e-07,0.629215,...,0,16697,0.0,0.0,0,0,0,1.995464e-08,8.940697e-08,2.976151e-08


,country,seed,accuracy,clipped_log_loss,resolved_farther_count,distance_tie_count,probability_tie_count,maximum_swap_error
0,Chile,101,0.680987,0.594979,0,3,0,8.940697e-08
1,Chile,201,0.684375,0.593649,0,4,1,1.043081e-07
2,Chile,301,0.686483,0.589048,0,0,0,1.192093e-07
3,Chile,401,0.682242,0.597383,0,61,59,1.192093e-07
4,Chile,501,0.685374,0.592766,0,66,64,8.940697e-08
5,France,301,0.627043,0.761073,0,2,0,1.192093e-07
6,Brazil,101,0.629215,0.743122,0,0,0,8.940697e-08
